# 03 · Fine-tune and evaluate an MLIP model

## Objective

This notebook uses the 10 GPAW/Chargemol labels from Notebook 02 to fine-tune the foundational MatQ model. It uses the calculated isolated-atom energies as references and trains the model for two epochs. Finally, the exported model is evaluated on the validation subset.

The small dataset and short training run show the procedure. They do not establish model accuracy.

## Prerequisites
Before execution, make sure the dependencies are met,

1. Complete all label and isolated-atom jobs in Notebook 02. Keep `examples/results/label-jobs.json` and the JobStore records.
2. Keep MongoDB running. Use the same JobStore configuration and isolated-atom registry as Notebook 02.
3. Open a terminal at the repository root. Install the notebook and MLIP dependencies:

   ```bash
   uv sync --group notebooks --group mlip
   ```

4. Make sure that `[programs].mlip_models_dir` in `crystal.toml` points to the `mlip_models` directory. 

GPAW and Chargemol are not required on the training computer. This notebook uses their stored results.

## 1. Import the training and evaluation tools

This step imports the training maker, dataset tools, and plotting helpers. Use the existing CRYSTAL implementations for record selection, data splitting, training, and evaluation.

In [ ]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from ase import Atoms
from ase.calculators.singlepoint import SinglePointCalculator
from ase.io import write
from jobflow.core.reference import OutputReference
from jobflow.managers.local import run_locally
from monty.serialization import dumpfn, loadfn
from pymatgen.io.ase import AseAtomsAdaptor

from crystal import CRYSTAL_ROOT
from crystal.calculators.mlip.qet_finetuning_dataset import (
    split_qet_fine_tuning_records,
)
from crystal.calculators.mlip.qet_finetuning_maker import (
    QetFineTuningMaker,
    collect_qet_fine_tuning_input_data,
)
from crystal.helpers.dataset import DatasetSummary
from crystal.helpers.dataset_plot_helpers import (
    plot_dataset_comparison_property_predictions,
)
from crystal.helpers.fireworks.connections import connect_to_jobstore
from crystal.helpers.model_evaluation_plot_helpers import (
    ModelEvaluationEntry,
    plot_model_comparison_parity_plots,
)
from crystal.helpers.model_evaluator import ModelEvaluator
from crystal.helpers.structure_summary import StructureSummary
from crystal.repositories.qet_finetuning_label_repository import (
    load_labelled_documents_for_qet_fine_tuning,
)
from crystal.schemas.gpaw_isolated_atom_reference import (
    GpawIsolatedAtomReferenceCollection,
    GpawRegistryIsolatedAtomReferenceSource,
)
from crystal.schemas.mlip_model_ref import MlipModelRef, QetModelRegistry
from crystal.schemas.properties import VectorMode
from crystal.schemas.qet_finetuning import (
    QetFineTuningLabelInput,
    QetFineTuningRecord,
    QetFineTuningResult,
    QetFineTuningSplitSettings,
    QetFineTuningStoredJobQuery,
    QetFineTuningTrainingSettings,
)


## 2. Set paths and prepare the training directory

This step selects the campaign file from Notebook 02 and sets the output paths. The campaign file records the label UUIDs, the isolated-atom collection UUID, and the JobStore configuration path.

**This cell deletes the previous `examples/results/training-runs` directory.** Copy any model or logs that you need before you run this cell. Each training job creates a new subdirectory. Clear the parent directory to prevent old checkpoints and models from accumulating. This does not delete the label files or JobStore records.

The result and training-input JSON files are overwritten after successful training. No training or evaluation flags are used.

In [ ]:
DATA_DIR = CRYSTAL_ROOT / "examples/results"
CAMPAIGN_PATH = DATA_DIR / "label-jobs.json"
TRAINING_PATH = DATA_DIR / "training-runs"
TRAINING_RESULT_PATH = DATA_DIR / "training-result.json"
TRAINING_INPUT_PATH = DATA_DIR / "training-input.json"
VALIDATION_PATH = DATA_DIR / "validation.extxyz"
MAX_EPOCHS = 2

if not CAMPAIGN_PATH.is_file():
    msg = f"No campaign file at {CAMPAIGN_PATH}. Complete Notebook 02 first."
    raise FileNotFoundError(msg)
if TRAINING_PATH.exists():
    shutil.rmtree(TRAINING_PATH)
TRAINING_PATH.mkdir(parents=True, exist_ok=True)


## 3. Retrieve the labels and reference energies

This step connects to the JobStore recorded by Notebook 02. Select only the campaign's label UUIDs at index 1. Use the existing repository loader so that inspection and training use the same record order. All requested labels must be present and usable.

**Local execution:** Connect to the JobStore here and pass it to `run_locally(..., store=jobstore)`. The training job requires this store to retrieve labels and resolve the isolated-atom registry. This notebook also retrieves the records here for data inspection and the validation split.

**FireWorks execution:** The worker uses its configured JobStore. You do not need this manual connection and retrieval step just to execute the training job. Submit with `manager.add_to_launchpad(job_or_flow=training_job)` and configure the worker to use the same label and registry stores. You still need the campaign's label query and reference-source settings to build the job. Retain the retrieval step if you want to run this notebook's inspection and evaluation cells.

Retrieve the isolated-atom collection through its UUID. Use its actual settings and eigensolver to select the physical energies from the registry during training. Keep the stored GPAW version and PAW provenance. Do not replace missing reference records with guessed energies.

In [ ]:
campaign = loadfn(CAMPAIGN_PATH)
if "jobstore_config" not in campaign:
    msg = f"Campaign {CAMPAIGN_PATH} has no JobStore configuration. Create a new campaign in Notebook 02."
    raise ValueError(msg)
JOBSTORE_CONFIG = Path(campaign["jobstore_config"])
jobstore = connect_to_jobstore(JOBSTORE_CONFIG)
label_source = QetFineTuningStoredJobQuery(
    criteria={"uuid": {"$in": campaign["labels"]}, "index": 1}
)
label_documents = load_labelled_documents_for_qet_fine_tuning(
    jobstore=jobstore,
    stored_job_query=label_source,
)
expected_label_count = len(campaign["labels"])
if len(label_documents) != expected_label_count:
    msg = f"Found {len(label_documents)} of {expected_label_count} labels. Complete or inspect Notebook 02's jobs."
    raise RuntimeError(msg)
input_data = collect_qet_fine_tuning_input_data(label_documents)
if input_data.failed_label_count:
    msg = f"The campaign has {input_data.failed_label_count} unusable labels. Inspect Notebook 02 before training."
    raise ValueError(msg)
atom_collection_uuid = campaign["atoms"]
atom_collection = GpawIsolatedAtomReferenceCollection.model_validate(
    OutputReference(atom_collection_uuid).resolve(jobstore)
)
if atom_collection.reference is None:
    msg = f"Isolated-atom collection {atom_collection_uuid} has no reference. Inspect its task documents."
    raise RuntimeError(msg)
reference = atom_collection.reference
reference_energies = reference.energies_by_element
eigensolvers = {entry.eigensolver for entry in reference.isolated_atom_energies}
reference_source = GpawRegistryIsolatedAtomReferenceSource(
    simulation_settings=reference.simulation_settings,
    eigensolver=next(iter(eigensolvers)) if len(eigensolvers) == 1 else None,
)
print("JobStore configuration:", JOBSTORE_CONFIG)
print(f"Stored labels: {len(label_documents)}; usable labels: {input_data.accepted_label_count}")
print("Calculated isolated-atom energies (eV):", reference_energies)


## 4. Configure training and split the data

This step selects the foundational MatQ model and sets two training epochs. Use an 80% training, 10% validation, and 10% test split with seed 42. Use the existing splitter with the same record order and settings as the training maker.

Ten labels give eight training records, one validation record, and one test record. Check that all subsets contain records. The validation record is used for the parity plots. The training runner evaluates the test subset separately after checkpoint selection.

In [ ]:
training_settings = QetFineTuningTrainingSettings(
    model=MlipModelRef(registry_model=QetModelRegistry.foundational_matq),
    batch_size=2,
    max_epochs=MAX_EPOCHS,
    split=QetFineTuningSplitSettings(
        train_fraction=0.8,
        validation_fraction=0.1,
        test_fraction=0.1,
        seed=42,
    ),
)
record_splits = split_qet_fine_tuning_records(
    input_data.accepted_records,
    training_settings.split,
)
split_counts = [
    len(record_splits.train_records),
    len(record_splits.validation_records),
    len(record_splits.test_records),
]
if not all(split_counts):
    msg = f"Training, validation, and test counts are {split_counts}. Generate more usable labels."
    raise ValueError(msg)
print("Training / validation / test:", split_counts)


## 5. Inspect the label distributions

This step builds a dataset summary and plots the energy, force, stress, and charge distributions. Use the existing plotting helper. Convert each stored record to ASE atoms with a single-point calculator so that the helper can read its labels. This does not run a new DFT calculation.

Calculate cohesive energy per atom for the plot with the stored isolated-atom references. Keep the original total-energy labels for training. Do not subtract the reference energies a second time.

In [ ]:
def labelled_atoms(record: QetFineTuningRecord) -> Atoms:
    atoms = AseAtomsAdaptor.get_atoms(record.structure)
    atoms.calc = SinglePointCalculator(
        atoms,
        energy=record.energy,
        forces=record.forces,
        stress=record.stress,
        charges=record.charges,
    )
    return atoms


summary = DatasetSummary(
    label="Notebook 02 GPAW campaign",
    structure_summaries=[
        StructureSummary.from_atoms(labelled_atoms(record))
        for record in input_data.accepted_records
    ],
    isolated_atom_energies=reference_energies,
)
_ = plot_dataset_comparison_property_predictions(
    [summary],
    force_mode=VectorMode.component,
    stress_mode=VectorMode.component,
)


## 6. Build and run the training job

This step builds the training job with `QetFineTuningMaker`. The maker retrieves the selected labels and resolves their physical references from the same JobStore and GPAW registry. It checks GPAW version, cutoff, XC, and PAW compatibility before training.

Run the job locally for two epochs. Wait for this cell to finish. The job writes its checkpoints, logs, and exported model below `TRAINING_PATH`. The runner selects the best validation-loss checkpoint for the exported model.

In [ ]:
training_job = QetFineTuningMaker(simulation_settings=training_settings).make(
    label_input=QetFineTuningLabelInput(label_source=label_source),
    isolated_atom_reference_source=reference_source,
)
run_locally(
    training_job,
    store=jobstore,
    root_dir=TRAINING_PATH,
    create_folders=True,
    ensure_success=True,
)


## 7. Retrieve and save the trained model reference

This step retrieves the training result from the JobStore. Check that training is complete before evaluation. Select the exported model, not the starting model. Save its result and the exact training inputs for later inspection. Existing files at these JSON paths are overwritten.

Notebook 05 can read `results/training-result.json` to use this model for geometry optimisation. Keep the exported model directory available. Running the directory-cleanup cell again deletes that model.

In [ ]:
training_result = QetFineTuningResult.model_validate(
    training_job.output.resolve(jobstore)
)
if not training_result.is_complete:
    msg = f"Training status is {training_result.status}. Inspect the run before evaluation."
    raise RuntimeError(msg)
if training_result.isolated_atom_reference is None:
    msg = f"Training job {training_job.uuid} has no calculated isolated-atom reference."
    raise ValueError(msg)
model_to_evaluate = training_result.model
reference_energies = training_result.isolated_atom_reference.energies_by_element
dumpfn(training_result, TRAINING_RESULT_PATH)
dumpfn(
    {
        "records": input_data.accepted_records,
        "settings": training_settings,
        "label_source": label_source,
        "atom_collection_uuid": campaign["atoms"],
        "jobstore_config": JOBSTORE_CONFIG.as_posix(),
    },
    TRAINING_INPUT_PATH,
)
print("Training job UUID:", training_job.uuid)
print("Exported model:", model_to_evaluate.model_path)
print("Saved training result:", TRAINING_RESULT_PATH)


## 8. Inspect training and validation loss

This step looks for CSV metrics in the current training output directory. If the logger wrote these metrics, plot the available training and validation loss by epoch. If no CSV loss history is available, inspect the Lightning logs in the training directory. This step does not start another training run.

In [ ]:
metric_files = sorted(
    training_result.checkpoint.path.parent.parent.glob("**/metrics.csv")
)
if metric_files:
    history = pd.read_csv(metric_files[-1])
    loss_columns = [
        name
        for name in ("train_Total_Loss", "val_Total_Loss")
        if name in history.columns
    ]
    if not history.empty and "epoch" in history.columns and loss_columns:
        loss_curves = history.groupby("epoch")[loss_columns].mean()
        _ = loss_curves.plot(
            marker="o",
            ylabel="Total loss",
            title="Training and validation loss",
        )
    else:
        print("The CSV file has no epoch loss metrics to plot. Inspect the Lightning logs.")
else:
    print("No CSV loss history is available. Inspect the Lightning logs in the training directory.")


## 9. Evaluate the model on the validation subset

This step writes only the validation records to `validation.extxyz`. The existing evaluator reads this file to compare stored labels with predictions from the trained model. The file is overwritten on each run.

Use the existing parity-plot helper for cohesive energy per atom, force components, stress components, and charges. Print MAE and RMSE with units. Save the metrics to `evaluation.json`.

A 10-label dataset has only one validation structure. These scores show the evaluation procedure, not reliable scientific accuracy. The training runner reports test metrics separately.

In [ ]:
write(
    VALIDATION_PATH,
    [labelled_atoms(record) for record in record_splits.validation_records],
    format="extxyz",
)
evaluation = ModelEvaluator(
    dataset_path=VALIDATION_PATH,
    isolated_atom_energies=reference_energies,
).evaluate(model=model_to_evaluate)
_ = plot_model_comparison_parity_plots(
    [ModelEvaluationEntry(result=evaluation, label="Validation")],
    force_mode=VectorMode.component,
    stress_mode=VectorMode.component,
)

metrics = []
for name, pair, unit in [
    ("Cohesive energy", evaluation.cohesive_energies_per_atom, "eV/atom"),
    ("Forces", evaluation.forces, "eV/Å"),
    ("Stress", evaluation.stress, "eV/Å³"),
    ("Charges", evaluation.charges, "e"),
]:
    metrics.append(
        {
            "property": name,
            "unit": unit,
            "MAE": float(np.mean(np.abs(pair.difference))),
            "RMSE": pair.rmse,
        }
    )
print(pd.DataFrame(metrics).to_string(index=False))
dumpfn(metrics, DATA_DIR / "evaluation.json")
